# XGBoost — Riesgo cardiovascular (BRFSS 2024)

Este cuaderno implementa el modelo XGBoost dentro del protocolo experimental descrito en la Sección IV del artículo. La primera fase valida el procedimiento sobre un único pliegue antes de ejecutar el ciclo completo de validación cruzada de 5 pliegues.

Los datos de entrada ya vienen imputados, normalizados y balanceados con SMOTENC, generados una sola vez por `scripts/preprocesar_pliegues.py` (ver `README` del proyecto). Si la carpeta `data_procesada/` no existe, es necesario correr ese script antes de continuar.

In [1]:
import os
import sys
sys.path.append("..")

import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

from scripts import pipeline as pl

DATA_DIR = "../data_procesada"
N_PLIEGUES = 5


def cargar(nombre):
    """Lee una particion ya preprocesada (generada por scripts/preprocesar_pliegues.py)."""
    tabla = pd.read_parquet(f"{DATA_DIR}/{nombre}")
    X = tabla.drop(columns=[pl.TARGET_COL])
    y = tabla[pl.TARGET_COL]
    return X, y

In [2]:
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        "No se encontro data_procesada/. Corre primero 'python scripts/preprocesar_pliegues.py' "
        "desde la raiz del proyecto (tarda unos 15 minutos, se corre una sola vez por maquina)."
    )

print("cache encontrado en", DATA_DIR)

cache encontrado en ../data_procesada


## Prueba sobre el pliegue 0

Antes de ejecutar el ciclo completo de 5 pliegues, se valida el procedimiento sobre un único pliegue, con el fin de confirmar que el modelo entrena correctamente sobre estos datos y que la función `pipeline.evaluar()` procesa sus predicciones sin errores.

In [3]:
X_train, y_train = cargar("fold_0_train.parquet")
X_test, y_test = cargar("fold_0_test.parquet")
prot_test = X_test[pl.PROTECTED_COLS]

print("train:", X_train.shape, "| balance:", y_train.value_counts().to_dict())

train: (634536, 30) | balance: {0: 317268, 1: 317268}


### Tratamiento de variables categóricas

Las variables categóricas (`_RACE`, `_INCOMG1`, entre otras) se mantienen codificadas como valores numéricos, sin asignarles el tipo `category` nativo de XGBoost ni aplicar codificación *one-hot*. XGBoost admite un tratamiento nativo de variables categóricas que podría mejorar su desempeño individual, pero dicho tratamiento no está disponible de forma equivalente en Regresión Logística ni en SVM. Dado que el objetivo del estudio es comparar los cuatro modelos bajo condiciones de preprocesamiento idénticas, se descarta ese tratamiento diferenciado para preservar la validez de la comparación.

In [4]:
modelo = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    random_state=pl.RANDOM_STATE,
    eval_metric="logloss",
)
modelo.fit(X_train, y_train)

y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]

In [5]:
resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)

print("RESUMEN (pliegue 0):")
for k, v in resumen.items():
    print(f"  {k}: {v:.4f}")

detalle["_RACE"]

RESUMEN (pliegue 0):
  auroc: 0.8032
  eod_sexo: 0.0951
  fpr_disp_sexo: 0.0245
  calib_desv_sexo: 0.4396
  eod_raza: 0.2407
  fpr_disp_raza: 0.0530
  calib_desv_raza: 0.6221
  eod_ingreso: 0.2615
  fpr_disp_ingreso: 0.0822
  calib_desv_ingreso: 0.5128


,grupo,n,tpr,fpr,calibracion
0,1.0,66555,0.284950,0.059406,1.426443
1,2.0,6987,0.114733,0.019614,1.161423
2,4.0,2469,0.044248,0.006367,1.327325
3,7.0,2053,0.116935,0.016066,0.929964
4,8.0,9544,0.123128,0.014537,1.622139
5,6.0,701,0.111111,0.008183,0.975073
6,3.0,1248,0.105263,0.019499,0.914051
7,5.0,395,0.106383,0.008621,0.866195


## Ajuste de hiperparámetros

Se realiza una búsqueda en grid sobre combinaciones de `n_estimators`, `max_depth` y `learning_rate`, con el fin de justificar la elección de hiperparámetros con evidencia, en vez de utilizar valores arbitrarios.

Para no comprometer los datos de prueba de ningún pliegue (lo cual constituiría fuga de datos), la búsqueda se realiza con una partición interna de validación tomada únicamente del train del pliegue 0 (80 % entrenamiento interno / 20 % validación interna), ya generada por `scripts/preprocesar_pliegues.py`. Los 5 pliegues oficiales reportados en la siguiente sección permanecen intactos durante todo este proceso.

In [6]:
X_tr_int, y_tr_int = cargar("tuning_train_interno.parquet")
X_val_int, y_val_int = cargar("tuning_validacion_interna.parquet")

print("train interno:", X_tr_int.shape, "| validacion interna:", X_val_int.shape)

train interno: (507628, 30) | validacion interna: (71962, 30)


In [7]:
grid = [
    dict(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate)
    for n_estimators in [100, 300]
    for max_depth in [4, 6, 8]
    for learning_rate in [0.05, 0.1]
]

resultados_grid = []
for params in grid:
    modelo_grid = XGBClassifier(
        **params, tree_method="hist", random_state=pl.RANDOM_STATE, eval_metric="logloss",
    )
    modelo_grid.fit(X_tr_int, y_tr_int)
    proba_val = modelo_grid.predict_proba(X_val_int)[:, 1]
    resultados_grid.append({**params, "auroc_validacion": roc_auc_score(y_val_int, proba_val)})

tabla_grid = pd.DataFrame(resultados_grid).sort_values("auroc_validacion", ascending=False)
tabla_grid.round(4)

,n_estimators,max_depth,learning_rate,auroc_validacion
11,300,8,0.10,0.8064
9,300,6,0.10,0.8047
10,300,8,0.05,0.8022
7,300,4,0.10,0.7995
8,300,6,0.05,0.7989
5,100,8,0.10,0.7980
3,100,6,0.10,0.7945
6,300,4,0.05,0.7927
4,100,8,0.05,0.7923
2,100,6,0.05,0.7904


La combinación con mejor AUROC de validación es `n_estimators=300, max_depth=8, learning_rate=0.1` (0.8064), por delante de `max_depth=6` (0.8047). A diferencia de la búsqueda realizada con SMOTE estándar (donde ambas profundidades quedaban casi empatadas), con SMOTENC la profundidad 8 gana con un margen más claro,  una diferencia atribuible a que SMOTENC genera ejemplos sintéticos menos "suavizados" en las variables categóricas (usa el valor más frecuente entre vecinos en vez de interpolar), por lo que el modelo necesita más profundidad para capturar los mismos patrones. Se observa además que `n_estimators=100` sigue quedando consistentemente por debajo de `n_estimators=300` en igualdad de los demás parámetros.

In [8]:
MEJORES_HIPERPARAMETROS = dict(n_estimators=300, max_depth=8, learning_rate=0.1)
MEJORES_HIPERPARAMETROS

{'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.1}

## Ciclo completo de validación cruzada (5 pliegues)

Una vez confirmado el funcionamiento sobre el pliegue 0, se repite el mismo procedimiento de forma independiente para cada uno de los 5 pliegues, y se reporta el promedio y la desviación estándar de cada métrica entre pliegues, tal como se especifica en la Sección IV-F del artículo.

In [9]:
resultados = []
detalle_total = []

for i in range(N_PLIEGUES):
    X_train, y_train = cargar(f"fold_{i}_train.parquet")
    X_test, y_test = cargar(f"fold_{i}_test.parquet")
    prot_test = X_test[pl.PROTECTED_COLS]

    modelo = XGBClassifier(
        **MEJORES_HIPERPARAMETROS,
        tree_method="hist",
        random_state=pl.RANDOM_STATE,
        eval_metric="logloss",
    )
    modelo.fit(X_train, y_train)

    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]

    resumen, detalle = pl.evaluar(y_test, y_pred, y_proba, prot_test)
    resumen["modelo"] = "xgboost"
    resumen["pliegue"] = i
    resultados.append(resumen)

    for col, tabla_grupo in detalle.items():
        tabla_grupo = tabla_grupo.copy()
        tabla_grupo["atributo"] = pl.SUFIJOS_PROTEGIDAS[col]
        tabla_grupo["modelo"] = "xgboost"
        tabla_grupo["pliegue"] = i
        detalle_total.append(tabla_grupo)

    print(f"pliegue {i} listo | AUROC = {resumen['auroc']:.4f}")

pliegue 0 listo | AUROC = 0.8057
pliegue 1 listo | AUROC = 0.8059
pliegue 2 listo | AUROC = 0.8082
pliegue 3 listo | AUROC = 0.8065
pliegue 4 listo | AUROC = 0.8091


In [10]:
tabla_metricas = pd.DataFrame(resultados)
columnas_ordenadas = ["modelo", "pliegue"] + [c for c in tabla_metricas.columns if c not in ("modelo", "pliegue")]
tabla_metricas = tabla_metricas[columnas_ordenadas]

resumen_cv = tabla_metricas.drop(columns=["modelo", "pliegue"]).agg(["mean", "std"]).round(4)

tabla_metricas.round(4)

,modelo,pliegue,auroc,eod_sexo,fpr_disp_sexo,calib_desv_sexo,eod_raza,fpr_disp_raza,calib_desv_raza,eod_ingreso,fpr_disp_ingreso,calib_desv_ingreso
0,xgboost,0,0.8057,0.0927,0.0225,0.3125,0.2090,0.0447,0.4354,0.2263,0.0748,0.3761
1,xgboost,1,0.8059,0.0798,0.0168,0.2927,0.2003,0.0457,0.3293,0.2054,0.0694,0.3515
2,xgboost,2,0.8082,0.0571,0.0144,0.3105,0.1933,0.0459,0.3032,0.2299,0.0707,0.3135
3,xgboost,3,0.8065,0.0714,0.0138,0.3009,0.2097,0.0437,0.4509,0.2251,0.0832,0.3495
4,xgboost,4,0.8091,0.0769,0.0152,0.2916,0.1903,0.0400,0.2858,0.1923,0.0692,0.2969


In [11]:
resumen_cv

,auroc,eod_sexo,fpr_disp_sexo,calib_desv_sexo,eod_raza,fpr_disp_raza,calib_desv_raza,eod_ingreso,fpr_disp_ingreso,calib_desv_ingreso
mean,0.8071,0.0756,0.0165,0.3017,0.2005,0.0440,0.3609,0.2158,0.0735,0.3375
std,0.0015,0.0130,0.0035,0.0097,0.0088,0.0024,0.0768,0.0162,0.0059,0.0318


**Nota metodológica**: estos resultados se obtuvieron con SMOTENC, no con SMOTE estándar. Al corregir el tratamiento de las variables categóricas (ver sección anterior), el AUROC promedio bajó de 0.8261 a 0.8071 (sigue por encima del baseline de literatura de 0.751–0.771), pero la disparidad entre grupos **aumentó**, por ejemplo, `eod_raza` pasó de 0.114 a 0.201. Una explicación plausible es que SMOTENC, al asignar a cada registro sintético el valor categórico más frecuente entre sus vecinos (en vez de interpolar numéricamente como SMOTE), genera ejemplos sintéticos menos diversos para los grupos demográficos minoritarios, lo cual podría amplificar su subrepresentación en el entrenamiento. Este hallazgo se documenta como parte de los resultados, ya que ilustra que corregir un problema metodológico (valores categóricos sin sentido) no necesariamente mejora todas las métricas a la vez.

## Guardado de resultados

Los resultados por pliegue se guardan en `results/metrics_xgboost.csv`, archivo que será consumido posteriormente por el cuaderno de comparación de los cuatro modelos.

In [12]:
import os

os.makedirs("../results", exist_ok=True)
tabla_metricas.to_csv("../results/metrics_xgboost.csv", index=False)
print("guardado en results/metrics_xgboost.csv")

guardado en results/metrics_xgboost.csv


## Detalle por grupo demográfico

Además del resumen anterior, se conserva el desglose por grupo (incluyendo el tamaño de muestra `n` de cada uno) para poder identificar grupos pequeños cuyas métricas sean menos estables, como algunas categorías de `_RACE` con pocos cientos de registros. Este detalle se guarda en `results/detalle_xgboost.csv`.

In [13]:
tabla_detalle = pd.concat(detalle_total, ignore_index=True)
tabla_detalle = tabla_detalle[["modelo", "pliegue", "atributo", "grupo", "n", "tpr", "fpr", "calibracion"]]

tabla_detalle.to_csv("../results/detalle_xgboost.csv", index=False)
print("guardado en results/detalle_xgboost.csv")

tabla_detalle.sort_values("n").head(10)

guardado en results/detalle_xgboost.csv


,modelo,pliegue,atributo,grupo,n,tpr,fpr,calibracion
9,xgboost,0,raza,5.0,395,0.170213,0.005747,0.855770
77,xgboost,4,raza,5.0,408,0.096154,0.016854,0.889066
60,xgboost,3,raza,5.0,408,0.173077,0.014045,0.889710
26,xgboost,1,raza,5.0,409,0.095238,0.002890,0.670692
43,xgboost,2,raza,5.0,433,0.083333,0.002519,1.222384
7,xgboost,0,raza,6.0,701,0.133333,0.016367,0.999575
40,xgboost,2,raza,6.0,740,0.129032,0.030912,1.076987
56,xgboost,3,raza,6.0,742,0.125000,0.029781,0.908838
72,xgboost,4,raza,6.0,758,0.145631,0.018321,0.960622
24,xgboost,1,raza,6.0,768,0.134021,0.019374,0.974204


## Comparación exploratoria: SMOTENC vs. class weighting

Esta sección es exploratoria y no forma parte del resultado principal reportado arriba. Se repite el mismo ciclo de 5 pliegues, con los mismos hiperparámetros, pero sustituyendo SMOTENC por `scale_pos_weight` (el equivalente de `class_weight="balanced"` en XGBoost), calculado en cada pliegue como la razón entre la clase mayoritaria y la minoritaria del train de ese pliegue. El objetivo es observar si una alternativa que no genera datos sintéticos produce resultados de equidad distintos a los de SMOTENC.

In [14]:
resultados_cw = []

for i in range(N_PLIEGUES):
    X_train, y_train = cargar(f"fold_{i}_train_sin_smotenc.parquet")
    X_test, y_test = cargar(f"fold_{i}_test.parquet")
    prot_test = X_test[pl.PROTECTED_COLS]

    conteo_clases = y_train.value_counts()
    scale_pos_weight = conteo_clases[0] / conteo_clases[1]

    modelo_cw = XGBClassifier(
        **MEJORES_HIPERPARAMETROS,
        scale_pos_weight=scale_pos_weight,
        tree_method="hist",
        random_state=pl.RANDOM_STATE,
        eval_metric="logloss",
    )
    modelo_cw.fit(X_train, y_train)

    y_pred = modelo_cw.predict(X_test)
    y_proba = modelo_cw.predict_proba(X_test)[:, 1]

    resumen_cw, _ = pl.evaluar(y_test, y_pred, y_proba, prot_test)
    resumen_cw["modelo"] = "xgboost_class_weight"
    resumen_cw["pliegue"] = i
    resultados_cw.append(resumen_cw)
    print(f"pliegue {i} listo | AUROC = {resumen_cw['auroc']:.4f} | scale_pos_weight = {scale_pos_weight:.2f}")

pliegue 0 listo | AUROC = 0.8180 | scale_pos_weight = 7.46
pliegue 1 listo | AUROC = 0.8162 | scale_pos_weight = 7.46
pliegue 2 listo | AUROC = 0.8186 | scale_pos_weight = 7.46
pliegue 3 listo | AUROC = 0.8160 | scale_pos_weight = 7.46
pliegue 4 listo | AUROC = 0.8178 | scale_pos_weight = 7.46


In [15]:
tabla_cw = pd.DataFrame(resultados_cw)
tabla_cw = tabla_cw[columnas_ordenadas]

tabla_cw.to_csv("../results/metrics_xgboost_class_weight.csv", index=False)
print("guardado en results/metrics_xgboost_class_weight.csv")

comparacion = pd.DataFrame({
    "smote": tabla_metricas.drop(columns=["modelo", "pliegue"]).mean(),
    "class_weight": tabla_cw.drop(columns=["modelo", "pliegue"]).mean(),
}).round(4)
comparacion

guardado en results/metrics_xgboost_class_weight.csv


,smote,class_weight
auroc,0.8071,0.8173
eod_sexo,0.0756,0.1269
fpr_disp_sexo,0.0165,0.0749
calib_desv_sexo,0.3017,1.9997
eod_raza,0.2005,0.3220
fpr_disp_raza,0.0440,0.2005
calib_desv_raza,0.3609,2.5050
eod_ingreso,0.2158,0.2585
fpr_disp_ingreso,0.0735,0.2761
calib_desv_ingreso,0.3375,2.2950


Con SMOTENC, el AUROC promedio de `class_weight` resulta ligeramente superior al de SMOTENC (0.8173 vs. 0.8071) — una diferencia pequeña pero ya no idéntica como ocurría con SMOTE estándar. La diferencia relevante sigue apareciendo en las métricas de equidad: con `scale_pos_weight`, el EOD y la disparidad de FPR aumentan en los tres atributos protegidos respecto a SMOTENC, y la desviación de calibración vuelve a dispararse de forma notable (por ejemplo, de 0.36 a 2.51 en raza). Esto es consistente con lo señalado en la literatura citada en la Sección II-B del artículo: `scale_pos_weight` corrige el desbalance inflando uniformemente las probabilidades predichas para la clase minoritaria, lo cual empeora la calibración, mientras que SMOTENC preserva una calibración considerablemente mejor al generar ejemplos sintéticos en vez de reponderar la función de pérdida. Con base en esto, se mantiene SMOTENC como técnica principal para los resultados reportados en este cuaderno, aceptando una pequeña pérdida de AUROC a cambio de probabilidades más confiables y de respetar la naturaleza categórica de los atributos protegidos.